## Benchmark pipeline demonstration

In [1]:
!pip install -q ultralytics

from ultralytics import YOLO
import pandas as pd
import numpy as np

from tqdm import tqdm
from collections import Counter
import yaml

import shutil
import glob
import os

real_images = glob.glob("/kaggle/input/datasets/paweekorns/sss-images/real_sonar/*/images/*")
synth_images = glob.glob("/kaggle/input/datasets/paweekorns/sss-images/synth_data/images/*")

print("Real image:", len(real_images))
print("synth_images:", len(synth_images))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Real image: 1149
synth_images: 1151


## Overview

In [2]:
shutil.copytree('/kaggle/input/datasets/paweekorns/sss-images/real_sonar', '/kaggle/working/real_sonar', dirs_exist_ok=True)
shutil.copytree('/kaggle/input/datasets/paweekorns/sss-images/synth_data', '/kaggle/working/synth_data', dirs_exist_ok=True)

with open("/kaggle/working/real_sonar/data.yaml", "r") as f:
    content = f.read()
    content = content.replace("./", "/kaggle/working/")
    f.close()

with open("/kaggle/working/real_sonar/data.yaml", "w") as f:
    f.write(content)

In [3]:
# check split ratio
dataset_path = '/kaggle/working/real_sonar'
def count_images(folder_name):
    count = 0
    folder_path = os.path.join(dataset_path, folder_name)
    
    # Recursively count files in common image formats
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif')):
                count += 1
    return count

# Get counts
train_count = count_images('train')
valid_count = count_images('valid')
test_count = count_images('test')

total = train_count + valid_count + test_count

if total > 0:
    print(f"Total Images: {total}")
    print(f"Train: {train_count} ({train_count/total:.2%})")
    print(f"Valid: {valid_count} ({valid_count/total:.2%})")
    print(f"Test:  {test_count} ({test_count/total:.2%})")
else:
    print("No images found. Please check your path.")

Total Images: 1149
Train: 802 (69.80%)
Valid: 113 (9.83%)
Test:  234 (20.37%)


In [4]:
yaml_path = os.path.join(dataset_path, 'data.yaml')
def get_distribution_by_class(dataset_root):
    # 1. Load config
    with open(yaml_path, 'r') as f:
        config = yaml.safe_load(f)
    
    class_names = config.get('names', [])
    splits = ['train', 'valid', 'test']
    
    # Structure: stats[class_id][subset] = count
    stats = {i: {s: 0 for s in splits} for i in range(len(class_names))}
    
    # 2. Count instances
    for split in splits:
        label_dir = os.path.join(dataset_root, split, 'labels')
        if not os.path.exists(label_dir):
            continue
            
        for filename in os.listdir(label_dir):
            if filename.endswith('.txt'):
                with open(os.path.join(label_dir, filename), 'r') as f:
                    for line in f:
                        try:
                            class_id = int(line.split()[0])
                            if class_id < len(class_names):
                                stats[class_id][split] += 1
                        except (ValueError, IndexError):
                            continue

    # 3. Create a clean list for the table
    results = []
    for cid, name in enumerate(class_names):
        counts = stats[cid]
        total = sum(counts.values())
        
        results.append({
            'Class Name': name,
            'Train': counts['train'],
            'Valid': counts['valid'],
            'Test': counts['test'],
            'Total': total,
            'Train (%)': round(counts['train'] / total, 2),
            'Valid (%)': round(counts['valid'] / total, 2),
            'Test (%)': round(counts['test'] / total, 2),
        })
    
    return pd.DataFrame(results)

# Run the analysis
df = get_distribution_by_class(dataset_root=dataset_path)
print(df.to_string(index=False))

Class Name  Train  Valid  Test  Total  Train (%)  Valid (%)  Test (%)
  cylinder    236     33    69    338       0.70       0.10       0.2
     manta     98     14    28    140       0.70       0.10       0.2
  airplane     74     10    21    105       0.70       0.10       0.2
 shipwreck    316     49    91    456       0.69       0.11       0.2
    victim    131     18    38    187       0.70       0.10       0.2


## Model Training

### Real data

In [5]:
model_id = "yolo11n"
model = YOLO(f"{model_id}.pt")

# Train the model
results = model.train(data="/kaggle/working/real_sonar/data.yaml", epochs=50, imgsz=640, batch=16)

Ultralytics 8.4.41 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/real_sonar/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patien

In [6]:
metrics = model.val(
    data='/kaggle/working/real_sonar/data.yaml', 
    split='test', 
    verbose=True
)

Ultralytics 8.4.41 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,583,127 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2620.1±1327.1 MB/s, size: 270.9 KB)
val: Scanning /kaggle/working/real_sonar/test/labels... 234 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 234/234 1.3Kit/s 0.2s
val: New cache created: /kaggle/working/real_sonar/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 5.0it/s 3.0s
                   all        234        247      0.728      0.664       0.73      0.403
              cylinder         69         69      0.739      0.451      0.546      0.241
                 manta         28         28      0.677      0.357      0.539       0.18
              airplane         21         21      0.725      0.762      0.836      0.502
             shipwreck         79         91      0.694      0

### Hybrid dataset (real+synthetic)

In [7]:
shutil.copytree('/kaggle/working/real_sonar', '/kaggle/working/merged_dataset', dirs_exist_ok=True)


dest_dir = "/kaggle/working/merged_dataset/train"
synth_dirs = ["/kaggle/working/synth_data"]

# add synthetic data into train set
for directory in synth_dirs:
    image_files = np.sort(glob.glob(os.path.join(directory, "images/*")))
    for img_path in tqdm(image_files):
        label_path = img_path.replace("images", "labels").replace(".jpg", ".txt").replace(".png", ".txt")
    
        dest_img_path = os.path.join(dest_dir, "images", f"{os.path.basename(img_path)}")
        dest_label_path = os.path.join(dest_dir, "labels", f"{os.path.basename(label_path)}")
    
        shutil.copy(img_path, dest_img_path)
        shutil.copy(label_path, dest_label_path)

# config data.yaml
with open("/kaggle/working/merged_dataset/data.yaml", "r") as f:
    content = f.read()
    content = content.replace("real_sonar", "merged_dataset")
    f.close()

with open("/kaggle/working/merged_dataset/data.yaml", "w") as f:
    f.write(content)

100%|██████████| 1151/1151 [00:00<00:00, 2848.98it/s]


In [8]:
hybrid_model = YOLO(f"{model_id}.pt")

# Train the model
results = hybrid_model.train(data="/kaggle/working/merged_dataset/data.yaml", epochs=50, imgsz=640, batch=16)

Ultralytics 8.4.41 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/merged_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, 

In [9]:
metrics = hybrid_model.val(
    data='/kaggle/working/real_sonar/data.yaml', 
    split='test', 
    verbose=True
)

Ultralytics 8.4.41 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,583,127 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2448.2±1040.5 MB/s, size: 169.4 KB)
val: Scanning /kaggle/working/real_sonar/test/labels.cache... 234 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 234/234 36.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 4.7it/s 3.2s
                   all        234        247      0.625      0.701      0.726      0.394
              cylinder         69         69      0.617      0.435      0.546      0.253
                 manta         28         28      0.596      0.527      0.554       0.21
              airplane         21         21      0.617       0.81       0.82      0.453
             shipwreck         79         91       0.66      0.894      0.845      0.463
                victim         37  